## Notebook to construct the test plant outage schedule

In [ ]:
import sys
sys.path.insert(0, '../../../src/CPM/')
from PertMain2 import Pert, Activity
import random
from datetime import datetime, time
import networkx as nx
import matplotlib.pyplot as plt

random.seed(0)

In [ ]:
# Main outage blocks
baseSchedule   = {'start': ['1','2','3'], 
                  '1': ['4','5','6','10'], 
                  '2': ['15','16'], 
                  '3': ['100','101','102','107','108'], 
                  '4': ['7'],
                  '5': ['7'],  
                  '6': ['8','9'],
                  '7': ['8'], 
                  '8': ['301'],
                  '9': ['301'],  

                  '10': ['11','12'], 
                  '11': ['13','14'],
                  '12': ['13','14'], 
                  '13': ['302'], 
                  '14': ['302'],
                  '15': ['17'],  
                  '16': ['17'],
                  '17': ['18','19','20'], 
                  '18': ['21','22'],
                  '19': ['21','22'],  
                  '20': ['21','22'], 
                  '21': ['302'],
                  '22': ['302'],

                  '100': ['103'], 
                  '101': ['103'], 
                  '102': ['105','106'], 
                  '103': ['104','105'], 
                  '104': ['109'],
                  '105': ['109'],  
                  '106': ['109'],
                  '107': ['110'], 
                  '108': ['110'],
                  '109': ['303'], 
                  '110': ['303'],  

                  '301': ['end'], 
                  '302': ['end'], 
                  '303': ['end'],                   

                  'end':[]}

Nmin = 1
Nmax = 3
numAct = {}
for act in baseSchedule.keys():
    if act in ['start','end']:
        numAct[act] = 1
    else:
        numAct[act] = random.randint(Nmin, Nmax)

fullActDict = {}
for act in baseSchedule.keys():
    if act not in ['start','end']:
        for i in range(numAct[act]):
            name = act +'-'+ str(i)
            duration = random.random()*10.0
            fullActDict[name] = Activity(name, duration)
            print(str(name) + ' = Activity("'+str(name)+'",'+str(duration)+', res={"res1":1})')
    elif act=='start':
        fullActDict['start'] = Activity('start', 1.)
        print('start = Activity("start",'+str(duration)+', res={"res1":1})')
    elif act=='end':
        fullActDict['end'] = Activity('end', 1.)
        print('end = Activity("end",'+str(duration)+', res={"res1":1})')
    else:
        print('error1') 

outageSchedule = {}
for act in baseSchedule.keys():
    if act=='end':
        outageSchedule[fullActDict['end']] = []
    else:
        for i in range(numAct[act]):
            if act=='start':
                name = 'start'
            else:
                name = act +'-'+ str(i)
                
            if i==numAct[act]-1:
                outageSchedule[fullActDict[name]] = []
                for desc in baseSchedule[act]:
                    if desc=='end':
                        succ = 'end'
                    else:
                        succ = desc +'-'+ str(0)
                    outageSchedule[fullActDict[name]].append(fullActDict[succ])
            else:
                succ = act +'-'+ str(i+1)
                outageSchedule[fullActDict[name]] = [fullActDict[succ]]

len(fullActDict.keys())

In [ ]:
outageStartTime =  datetime(2025, 4, 25, 8)

pert = Pert(outageSchedule, startTime=outageStartTime)

In [ ]:
pert.getCriticalPathSymbolic()

In [ ]:
pert.returnScheduleEndTime()

In [ ]:
def plotSchedule(outageSchedule):
    # Create a directed graph
    G = nx.DiGraph()

    # Add edges
    for node, neighbors in outageSchedule.items():
        for neighbor in neighbors:
            G.add_edge(node, neighbor)

    # Assign layers based on distance from 'start'
    layers = {}
    def assign_layers(node, depth=0):
        if node in layers:
            layers[node] = max(layers[node], depth)
        else:
            layers[node] = depth
        for neighbor in outageSchedule.get(node, []):
            assign_layers(neighbor, depth + 1)

    assign_layers('start')

    # Create a layout based on layers
    pos = {}
    layer_nodes = {}
    for node, layer in layers.items():
        layer_nodes.setdefault(layer, []).append(node)

    for layer, nodes in layer_nodes.items():
        for i, node in enumerate(nodes):
            pos[node] = (layer, -i)

    # Draw the graph
    plt.figure(figsize=(20, 12))
    nx.draw(G, pos, with_labels=True, node_size=500, node_color='lightgreen', font_size=10, arrows=True)
    plt.title("Course Schedule Graph with 'start' on the Left and 'end' on the Right", fontsize=16)
    plt.axis('off')
    plt.show()


In [ ]:
plotSchedule(pert.returnGraphSymbolic())

In [ ]:
pert.returnGraphSymbolic()